<a href="https://colab.research.google.com/github/Emmanuel-NS/FARUMASI-AI/blob/main/notebooks/train_chat_reader_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chat reader: Qwen2.5-1.5B-Instruct + LoRA

Runtime → Change runtime type → **T4 GPU**. Run the cells in order (about 1.5–2 hours in total).

Fine-tunes Qwen2.5-1.5B-Instruct with LoRA to write pharmacist-style answers grounded on the retrieved passages (direct first sentence, then details, and "my sources don't cover this" when the passages do not answer). Then compares it with the zero-shot model on the same test questions as the Flan-T5 reader.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "https://github.com/Emmanuel-NS/FARUMASI-AI.git"
%cd /content
!rm -rf repo && git clone -q $REPO_URL repo
%cd /content/repo
!pip install -q -r requirements.txt

## 1. Data (same corpus and splits as the main notebook)

In [ ]:
!python -m pharmqa.data.download
!python -m pharmqa.data.prepare

## 2. LoRA fine-tuning (about 60–80 min on a T4)

3,000 answer examples + up to 400 multi-section overviews + about 8% "no answer" examples, 1 epoch. If Colab disconnects, re-run the cell with `--resume` added.

In [ ]:
!python -m pharmqa.generation.train_chat_lora --epochs 1 --max_train 3000 --batch 2 --grad_accum 8

## 3. Evaluation (about 20–30 min)

Gold context = the right passage + 2 hard negatives (same condition as the Flan-T5 table). Retrieved context falls back to BM25 here because the fine-tuned retriever is not trained in this notebook.

In [ ]:
for system in ["zeroshot", "lora"]:
    !python -m pharmqa.generation.evaluate_chat --system {system} --context gold --n 200

In [ ]:
!python -m pharmqa.generation.evaluate_chat --system lora --context retrieved --n 200 --tag _bm25

## 4. Download the adapter and results

Unzip `chat_reader.zip` into the project folder on your computer (it creates `artifacts/generator_qwen_lora/` and `results/chat_*`).

In [ ]:
!rm -rf artifacts/generator_qwen_lora/checkpoints
!zip -qr chat_reader.zip artifacts/generator_qwen_lora results/chat_*
from google.colab import files
files.download("chat_reader.zip")